# Numpy Storage Format
#### Erik S. Sørensen 2.2.2021
##### Adapted from a stackoverflow.com post by Gareth Rees Nov 2017

In [57]:
import numpy as np
a = np.arange(12)
a

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

How is this actually stored ? Core python would store an array as a list

In [58]:
x=[1,2,4,56,6]
type(x)

list

In [59]:
y=(1.0,3.0)
type(y)

tuple

In [60]:
z={3.0,4.0}
type(z)

set

In Numpy things are stored differently. If we delcare the numpy array **a** as above
then a consists of a **data buffer**, arranged something like this:

┌────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┐
│  0 │  1 │  2 │  3 │  4 │  5 │  6 │  7 │  8 │  9 │ 10 │ 11 │
└────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┘

But also additional data specifying how the array should be the **view**

In [61]:
a.flags

  C_CONTIGUOUS : True
  F_CONTIGUOUS : True
  OWNDATA : True
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False
  UPDATEIFCOPY : False

In [62]:
a.dtype

dtype('int64')

In [63]:
a.shape

(12,)

Here the shape (12,) means the array is indexed by a single index which runs from 0 to 11. Conceptually, if we label this single index **i**, the array **a** looks like this:

i= 0    1    2    3    4    5    6    7    8    9   10   11
┌────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┐
│  0 │  1 │  2 │  3 │  4 │  5 │  6 │  7 │  8 │  9 │ 10 │ 11 │
└────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┘

If we **reshape** an array, this doesn't change the data buffer. Instead, it creates a new view that describes a different way to interpret the data. So after:

In [64]:
b = a.reshape((3, 4))

the array **b** has a copy of the same data buffer as **a**, but now it is indexed by two indices which run from 0 to 2 and 0 to 3 respectively. If we label the two indices **i** and **j**, the array **b** looks like this:

i= 0    0    0    0    1    1    1    1    2    2    2    2
j= 0    1    2    3    0    1    2    3    0    1    2    3
┌────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┐
│  0 │  1 │  2 │  3 │  4 │  5 │  6 │  7 │  8 │  9 │ 10 │ 11 │
└────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┘

which means that:

In [65]:
b[2,1]

9

You can see that the second index changes quickly and the first index changes slowly. This is the way a matrix would be stored in **C**. In **Fortran** it is the precise other way. If you prefer this to the **C** way, you can specify the **order** parameter:

In [66]:
c = a.reshape((3, 4), order='F')

which results in an array indexed like this:

i= 0    1    2    0    1    2    0    1    2    0    1    2
j= 0    0    0    1    1    1    2    2    2    3    3    3
┌────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┐
│  0 │  1 │  2 │  3 │  4 │  5 │  6 │  7 │  8 │  9 │ 10 │ 11 │
└────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┘

which means that:

In [67]:
c[2,1]

5

It should now be clear what it means for an array to have a shape with one or more dimensions of size 1. After:

In [69]:
d = a.reshape((12, 1))

the array **d** is indexed by two indices, the first of which runs from 0 to 11, and the second index is always 0:

i= 0    1    2    3    4    5    6    7    8    9   10   11
j= 0    0    0    0    0    0    0    0    0    0    0    0
┌────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┐
│  0 │  1 │  2 │  3 │  4 │  5 │  6 │  7 │  8 │  9 │ 10 │ 11 │
└────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┘

## This corresponds to a column vector

and so:

In [70]:
d[10,0]

10

A dimension of length 1 is "free" (in some sense), so there's nothing stopping you from going to town:

In [71]:
e = a.reshape((1, 2, 1, 6, 1))

giving an array indexed like this:

i= 0    0    0    0    0    0    0    0    0    0    0    0
j= 0    0    0    0    0    0    1    1    1    1    1    1
k= 0    0    0    0    0    0    0    0    0    0    0    0
l= 0    1    2    3    4    5    0    1    2    3    4    5
m= 0    0    0    0    0    0    0    0    0    0    0    0
┌────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┬────┐
│  0 │  1 │  2 │  3 │  4 │  5 │  6 │  7 │  8 │  9 │ 10 │ 11 │
└────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┴────┘

and so:

In [72]:
e[0,1,0,0,0]

6

# Lesson 2: A column vector has shape (n,1) or (n,) a row vector has shape (1,n)

Now back to our initial example of the array **a**: Why is it that the shape of the array is (12,)

In [73]:
a = np.arange(12)
a.shape

(12,)

The short answer is that the data type for the shape has to be a tuple and (12) is not a tuple only (12,) is. The other point is that it is not clear if the array of twelve elements should be seen as a row vector or a column vector and if we had used (12) for the shape it would have been ambiguous. Hence numpy makes the decision for us and force the vector to be a column vector. Technically, there is not a second index in the view as there would have been with a shape of (12,1). However, we can force this:

In [74]:
a.shape=(12,1)
a.shape

(12, 1)

However, we can use an array with shape (12,) as a column vector without problem although that is probably not to be recommended:

In [75]:
b = np.arange(12)
c = np.arange(144)
c.shape=(12,12)
c.shape

(12, 12)

In [76]:
d=np.dot(c,b)
print(d)

[ 506 1298 2090 2882 3674 4466 5258 6050 6842 7634 8426 9218]


In [77]:
d.shape

(12,)

## Reshaping with -1

Sometimes we might have an array that we want to reshape into a number of rows but we can't be bother to calculate how many columns that would correspond to. We can then just let numpy do the calculation:

In [78]:
f=np.arange(256)
print(f.shape)
f.shape=(8,-1)
print(f.shape)

(256,)
(8, 32)


In [79]:
f.shape=(-1,8)
print(f.shape)

(32, 8)


In [80]:
f.shape=(-1)
print(f.shape)

(256,)


In the last step we have **flattened** the array

However, I cannot reshape the array into something that doesn't fit with the number of elements:

In [81]:
f=np.arange(256)
print(f.shape)
f.shape=(13,-1)
print(f.shape)

(256,)


ValueError: cannot reshape array of size 256 into shape (13,newaxis)

# Lesson 3: The number of elements in the data buffer of an array of shape (i,j,k,l) MUST be i x j x k x l